In [8]:
import numpy as np
import pandas as pd
from meteostat import daily
import pandas as pd
from datetime import date


In [9]:

weights_path = "../EDA/dep_station_KNN_weights.csv"

w = pd.read_csv(
    weights_path,
    dtype={"departement_code": str, "station_id": str}
)

# Ensure correct structure
assert {"departement_code", "station_id", "weight"}.issubset(w.columns)

# Build dep -> weights Series
dep_weights = {
    dep: g.set_index("station_id")["weight"].astype(float)
    for dep, g in w.groupby("departement_code")
}

print("Number of departments:", len(dep_weights))


Number of departments: 109


In [11]:
import meteostat as ms
import pandas as pd
from datetime import date

station_ids = sorted(w["station_id"].unique())

start = date(2000, 1, 1)
end   = date(2024, 12, 31)

dfs = []
failed = []
empty = []

for sid in station_ids:
    try:
        station = ms.Station(id=sid)
        ts = ms.daily(station, start, end)
        df = ts.fetch()

        if df is None or df.empty:
            empty.append(sid)
            continue

        df = df.reset_index()  # index 'time' becomes column
        df["station_id"] = sid

        # Your fetch() output shows columns: temp, tmin, tmax, ...
        keep = ["time", "station_id"]
        for c in ["temp", "tmin", "tmax"]:
            if c in df.columns:
                keep.append(c)

        dfs.append(df[keep])

    except Exception as e:
        failed.append((sid, repr(e)))

print("Fetched stations:", len(dfs))
print("Empty stations:", len(empty))
print("Failed stations:", len(failed))

# Show a few failures (if any)
print("Example failures:", failed[:10])
print("Example empty:", empty[:10])

if len(dfs) == 0:
    raise RuntimeError("No station data fetched. Check example failures above.")

station_long = pd.concat(dfs, ignore_index=True)

station_daily_temp = (
    station_long
    .rename(columns={"time": "date"})
    .pivot(index="date", columns="station_id", values="temp")
    .sort_index()
)

station_daily_temp.index = pd.to_datetime(station_daily_temp.index)
station_daily_temp.columns.name = "station_id"

print("station_daily_temp shape:", station_daily_temp.shape)
station_daily_temp.head()


Fetched stations: 204
Empty stations: 12
Failed stations: 0
Example failures: []
Example empty: ['07033', '07148', '07186', '07276', '07362', '07517', '07657', '99854', 'LFFS0', 'LFNB0']
station_daily_temp shape: (9132, 204)


station_id,07002,07003,07005,07010,07015,07017,07024,07027,07028,07029,...,LFMU0,LFOQ0,LFOV0,LFOZ0,LFPT0,LFQA0,LFRU0,LFRV0,LFSG0,LFSM0
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,8.3,<NA>,8.4,7.9,7.3,7.0,<NA>,9.3,8.2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-02,8.7,<NA>,9.1,9.0,8.2,8.0,<NA>,10.2,9.2,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-03,8.7,<NA>,9.4,9.7,8.6,7.7,<NA>,9.8,8.8,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-04,8.4,<NA>,8.6,8.0,8.2,7.9,<NA>,9.7,8.7,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-05,7.0,<NA>,6.2,6.4,4.8,<NA>,<NA>,7.3,7.5,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Using regions 

In [12]:

weights_path = "../EDA/reg_station_KNN_weights.csv"

w = pd.read_csv(
    weights_path,
    dtype={"departement_code": str, "station_id": str}
)

# Ensure correct structure
assert {"departement_code", "station_id", "weight"}.issubset(w.columns)

# Build dep -> weights Series
dep_weights = {
    dep: g.set_index("station_id")["weight"].astype(float)
    for dep, g in w.groupby("departement_code")
}

print("Number of Regions:", len(dep_weights))


Number of Regions: 26


In [13]:
import meteostat as ms
import pandas as pd
from datetime import date

station_ids = sorted(w["station_id"].unique())

start = date(2000, 1, 1)
end   = date(2024, 12, 31)

dfs = []
failed = []
empty = []

for sid in station_ids:
    try:
        station = ms.Station(id=sid)
        ts = ms.daily(station, start, end)
        df = ts.fetch()

        if df is None or df.empty:
            empty.append(sid)
            continue

        df = df.reset_index()  # index 'time' becomes column
        df["station_id"] = sid

        # Your fetch() output shows columns: temp, tmin, tmax, ...
        keep = ["time", "station_id"]
        for c in ["temp", "tmin", "tmax"]:
            if c in df.columns:
                keep.append(c)

        dfs.append(df[keep])

    except Exception as e:
        failed.append((sid, repr(e)))

print("Fetched stations:", len(dfs))
print("Empty stations:", len(empty))
print("Failed stations:", len(failed))

# Show a few failures (if any)
print("Example failures:", failed[:10])
print("Example empty:", empty[:10])

if len(dfs) == 0:
    raise RuntimeError("No station data fetched. Check example failures above.")

station_long = pd.concat(dfs, ignore_index=True)

station_daily_temp = (
    station_long
    .rename(columns={"time": "date"})
    .pivot(index="date", columns="station_id", values="temp")
    .sort_index()
)

station_daily_temp.index = pd.to_datetime(station_daily_temp.index)
station_daily_temp.columns.name = "station_id"

print("station_daily_temp shape:", station_daily_temp.shape)
station_daily_temp.head()


Fetched stations: 80
Empty stations: 5
Failed stations: 0
Example failures: []
Example empty: ['07148', '07186', '07276', '07657', '99854']
station_daily_temp shape: (9132, 80)


station_id,07015,07017,07027,07028,07029,07031,07046,07059,07061,07090,...,91754,LFAQ0,LFBU0,LFJR0,LFLH0,LFLU0,LFOQ0,LFOV0,LFOZ0,LFRV0
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,7.3,7.0,9.3,8.2,<NA>,<NA>,<NA>,<NA>,6.4,4.3,...,26.6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-02,8.2,8.0,10.2,9.2,<NA>,<NA>,<NA>,<NA>,7.3,4.5,...,26.1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-03,8.6,7.7,9.8,8.8,<NA>,<NA>,<NA>,<NA>,6.9,2.9,...,25.8,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-04,8.2,7.9,9.7,8.7,<NA>,<NA>,<NA>,<NA>,7.5,6.7,...,25.6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-05,4.8,<NA>,7.3,7.5,<NA>,<NA>,<NA>,<NA>,4.6,7.1,...,28.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [14]:
# ---------- 1) Department aggregation (your KNN weights) ----------
def dept_daily_temp(station_daily_tavg: pd.DataFrame, dep_weight_series: pd.Series) -> pd.Series:
    common = [sid for sid in dep_weight_series.index if sid in station_daily_tavg.columns]
    X = station_daily_tavg[common]
    W = dep_weight_series.loc[common]

    avail = ~X.isna()
    W_avail = avail.mul(W, axis=1)
    denom = W_avail.sum(axis=1).replace(0, np.nan)

    T_dep = (X.fillna(0) * W_avail).sum(axis=1) / denom
    return T_dep

# ---------- 2) Seasonality fit ----------
def add_time_features(df, date_col="date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    df["doy"] = df[date_col].dt.dayofyear.astype(int)
    df["sin1"] = np.sin(2 * np.pi * df["doy"] / 365.25)
    df["cos1"] = np.cos(2 * np.pi * df["doy"] / 365.25)
    df["t"] = (df[date_col] - df[date_col].min()).dt.days.astype(float)
    return df

def fit_seasonal_mean(df, temp_col="temp", include_trend=True):
    X_cols = ["sin1", "cos1"]
    if include_trend:
        X_cols = ["t"] + X_cols

    X = np.column_stack([np.ones(len(df))] + [df[c].values for c in X_cols])
    y = df[temp_col].values
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    S = X @ beta

    def seasonal_mean_func(new_dates: pd.DatetimeIndex):
        new_df = pd.DataFrame({"date": new_dates})
        new_df = add_time_features(new_df, "date")
        Xn = np.column_stack([np.ones(len(new_df))] + [new_df[c].values for c in X_cols])
        return Xn @ beta

    return beta, S, seasonal_mean_func

# ---------- 3) OU (AR1) fit on residuals ----------
def fit_ou_ar1(residuals):
    x = residuals[:-1]
    y = residuals[1:]
    phi = np.dot(x, y) / np.dot(x, x)
    phi = np.clip(phi, 1e-6, 0.999999)

    eps = y - phi * x
    sigma_eps = eps.std(ddof=1)
    kappa = -np.log(phi)  # Δ=1 day

    return phi, sigma_eps, kappa

# ---------- 4) Simulation ----------
def simulate_temperatures(dates, seasonal_mean_func, phi, sigma_eps, x0, n_paths=20000, seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.to_datetime(dates)
    n = len(dates)

    S = seasonal_mean_func(dates)

    X = np.empty((n_paths, n), dtype=float)
    T = np.empty((n_paths, n), dtype=float)

    X[:, 0] = x0
    T[:, 0] = S[0] + X[:, 0]

    shocks = rng.normal(0.0, sigma_eps, size=(n_paths, n - 1))
    for t in range(1, n):
        X[:, t] = phi * X[:, t - 1] + shocks[:, t - 1]
        T[:, t] = S[t] + X[:, t]

    return T  # shape: (paths, days)

# ---------- 5) Indices ----------
def CAT_index(T_paths):
    return T_paths.sum(axis=1)

def HDD_index(T_paths, base=10.0):
    return np.maximum(base - T_paths, 0.0).sum(axis=1)

# ---------- 6) Pricing ----------
def price_future(index_samples, r=0.0, tau_years=0.0):
    return np.exp(-r * tau_years) * index_samples.mean()

def price_call(index_samples, strike, notional=1.0, r=0.0, tau_years=0.0):
    payoff = np.maximum(index_samples - strike, 0.0) * notional
    return np.exp(-r * tau_years) * payoff.mean()

def price_put(index_samples, strike, notional=1.0, r=0.0, tau_years=0.0):
    payoff = np.maximum(strike - index_samples, 0.0) * notional
    return np.exp(-r * tau_years) * payoff.mean()

# ---------- 7) One department: fit + simulate + price ----------
def ou_price_department(
    station_daily_tavg, dep_weight_series,
    window_start, window_end,
    hdd_base=10.0,
    include_trend=True,
    n_paths=50000,
    seed=0,
    r=0.0
):
    # Build dept daily temp
    T_dep = dept_daily_temp(station_daily_tavg, dep_weight_series).dropna()
    df = pd.DataFrame({"date": T_dep.index, "temp": T_dep.values})
    df = add_time_features(df, "date")

    # Fit seasonal mean + OU
    _, S, seasonal_mean_func = fit_seasonal_mean(df, "temp", include_trend=include_trend)
    resid = df["temp"].values - S
    phi, sigma_eps, kappa = fit_ou_ar1(resid)

    # Simulate for the contract window
    dates = pd.date_range(pd.to_datetime(window_start), pd.to_datetime(window_end), freq="D")
    x0 = resid[-1]  # last observed residual
    T_paths = simulate_temperatures(dates, seasonal_mean_func, phi, sigma_eps, x0, n_paths=n_paths, seed=seed)

    # Build indices
    cat = CAT_index(T_paths)
    hdd = HDD_index(T_paths, base=hdd_base)

    tau_years = len(dates) / 365.25
    cat_fut = price_future(cat, r=r, tau_years=tau_years)
    hdd_fut = price_future(hdd, r=r, tau_years=tau_years)

    return {
        "phi": phi, "kappa": kappa, "sigma_eps": sigma_eps,
        "CAT_mean": float(cat.mean()), "CAT_std": float(cat.std(ddof=1)), "CAT_future": float(cat_fut),
        "HDD_mean": float(hdd.mean()), "HDD_std": float(hdd.std(ddof=1)), "HDD_future": float(hdd_fut),
    }


In [16]:

# =========================
# 0) LOAD WEIGHTS
# =========================
weights_path = "../EDA/reg_station_KNN_weights.csv"

w = pd.read_csv(
    weights_path,
    dtype={"departement_code": str, "station_id": str}
)

assert {"departement_code", "station_id", "weight"}.issubset(w.columns)

dep_weights = {
    dep: g.set_index("station_id")["weight"].astype(float)
    for dep, g in w.groupby("departement_code")
}
print("Number of Regions:", len(dep_weights))

# =========================
# 1) PULL STATION DATA
# =========================
station_ids = sorted(w["station_id"].unique())

start = date(2000, 1, 1)
end   = date(2024, 12, 31)

dfs, failed, empty = [], [], []

for sid in station_ids:
    try:
        station = ms.Station(id=sid)
        ts = ms.daily(station, start, end)
        df = ts.fetch()

        if df is None or df.empty:
            empty.append(sid)
            continue

        df = df.reset_index()  # index 'time' -> column
        df["station_id"] = sid

        keep = ["time", "station_id"]
        for c in ["temp", "tmin", "tmax"]:
            if c in df.columns:
                keep.append(c)

        dfs.append(df[keep])

    except Exception as e:
        failed.append((sid, repr(e)))

print("Fetched stations:", len(dfs))
print("Empty stations:", len(empty))
print("Failed stations:", len(failed))
print("Example failures:", failed[:10])
print("Example empty:", empty[:10])

if len(dfs) == 0:
    raise RuntimeError("No station data fetched. Check example failures above.")

station_long = pd.concat(dfs, ignore_index=True)

station_daily_temp = (
    station_long
    .rename(columns={"time": "date"})
    .pivot(index="date", columns="station_id", values="temp")
    .sort_index()
)

station_daily_temp.index = pd.to_datetime(station_daily_temp.index)
station_daily_temp.columns.name = "station_id"

# Ensure numeric (convert any <NA>/objects to NaN)
station_daily_temp = station_daily_temp.apply(pd.to_numeric, errors="coerce")

print("station_daily_temp shape:", station_daily_temp.shape)
display(station_daily_temp.head())

# =========================
# 2) DEPARTMENT/REGION AGGREGATION
# =========================
def dept_daily_temp(station_daily: pd.DataFrame, dep_weight_series: pd.Series) -> pd.Series:
    """
    Weighted average with daily renormalization for missing station values.
    """
    common = [sid for sid in dep_weight_series.index if sid in station_daily.columns]
    if len(common) == 0:
        return pd.Series(index=station_daily.index, dtype=float)

    X = station_daily[common]
    W = dep_weight_series.loc[common].astype(float)

    avail = ~X.isna()
    W_avail = avail.mul(W, axis=1)
    denom = W_avail.sum(axis=1).replace(0, np.nan)

    T_dep = (X.fillna(0) * W_avail).sum(axis=1) / denom
    return T_dep








Number of Regions: 26
Fetched stations: 80
Empty stations: 5
Failed stations: 0
Example failures: []
Example empty: ['07148', '07186', '07276', '07657', '99854']
station_daily_temp shape: (9132, 80)


station_id,07015,07017,07027,07028,07029,07031,07046,07059,07061,07090,...,91754,LFAQ0,LFBU0,LFJR0,LFLH0,LFLU0,LFOQ0,LFOV0,LFOZ0,LFRV0
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,7.3,7.0,9.3,8.2,<NA>,<NA>,<NA>,<NA>,6.4,4.3,...,26.6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-02,8.2,8.0,10.2,9.2,<NA>,<NA>,<NA>,<NA>,7.3,4.5,...,26.1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-03,8.6,7.7,9.8,8.8,<NA>,<NA>,<NA>,<NA>,6.9,2.9,...,25.8,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-04,8.2,7.9,9.7,8.7,<NA>,<NA>,<NA>,<NA>,7.5,6.7,...,25.6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-01-05,4.8,<NA>,7.3,7.5,<NA>,<NA>,<NA>,<NA>,4.6,7.1,...,28.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [17]:
# =========================
# 3) SEASONALITY FIT
# =========================
def add_time_features(df, date_col="date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    df["doy"] = df[date_col].dt.dayofyear.astype(int)
    df["sin1"] = np.sin(2 * np.pi * df["doy"] / 365.25)
    df["cos1"] = np.cos(2 * np.pi * df["doy"] / 365.25)
    df["t"] = (df[date_col] - df[date_col].min()).dt.days.astype(float)
    return df

def fit_seasonal_mean(df, temp_col="temp", include_trend=True):
    X_cols = ["sin1", "cos1"]
    if include_trend:
        X_cols = ["t"] + X_cols

    X = np.column_stack([np.ones(len(df))] + [df[c].values for c in X_cols])
    y = df[temp_col].values.astype(float)

    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    S = X @ beta

    def seasonal_mean_func(new_dates: pd.DatetimeIndex):
        new_df = pd.DataFrame({"date": pd.to_datetime(new_dates)})
        new_df = add_time_features(new_df, "date")
        Xn = np.column_stack([np.ones(len(new_df))] + [new_df[c].values for c in X_cols])
        return Xn @ beta

    return beta, S, seasonal_mean_func


In [18]:
# =========================
# 4) OU (AR1) FIT
# =========================
def fit_ou_ar1(residuals):
    residuals = np.asarray(residuals, dtype=float)
    if len(residuals) < 30:
        raise ValueError("Not enough residual data to fit AR(1) reliably (need >= 30).")

    x = residuals[:-1]
    y = residuals[1:]

    denom = np.dot(x, x)
    if denom <= 1e-12:
        raise ValueError("Residual variance too small; cannot fit AR(1).")

    phi = np.dot(x, y) / denom
    phi = np.clip(phi, 1e-6, 0.999999)

    eps = y - phi * x
    sigma_eps = float(np.std(eps, ddof=1))
    kappa = float(-np.log(phi))  # Δ=1 day

    return float(phi), sigma_eps, kappa

In [19]:
# =========================
# 5) SIMULATION
# =========================
def simulate_temperatures(dates, seasonal_mean_func, phi, sigma_eps, x0, n_paths=20000, seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.to_datetime(dates)
    n = len(dates)

    S = seasonal_mean_func(dates)

    X = np.empty((n_paths, n), dtype=float)
    T = np.empty((n_paths, n), dtype=float)

    X[:, 0] = x0
    T[:, 0] = S[0] + X[:, 0]

    shocks = rng.normal(0.0, sigma_eps, size=(n_paths, n - 1))
    for t in range(1, n):
        X[:, t] = phi * X[:, t - 1] + shocks[:, t - 1]
        T[:, t] = S[t] + X[:, t]

    return T


In [20]:
# =========================
# 6) INDICES + PRICING
# =========================
def CAT_index(T_paths):
    return T_paths.sum(axis=1)

def HDD_index(T_paths, base=10.0):
    return np.maximum(base - T_paths, 0.0).sum(axis=1)

def price_future(index_samples, r=0.0, tau_years=0.0):
    return float(np.exp(-r * tau_years) * np.mean(index_samples))



In [21]:
# =========================
# 7) ONE REGION: FIT + SIM + PRICE
# =========================
def ou_price_department(
    station_daily, dep_weight_series,
    window_start, window_end,
    hdd_base=10.0,
    include_trend=True,
    n_paths=50000,
    seed=0,
    r=0.0
):
    # Build dept daily temp
    T_dep = dept_daily_temp(station_daily, dep_weight_series).dropna()
    if len(T_dep) < 365:  # at least ~1 year of daily data
        raise ValueError("Too few department temperature observations after aggregation.")

    df = pd.DataFrame({"date": T_dep.index, "temp": T_dep.values})
    df = add_time_features(df, "date")

    # Fit seasonal mean + OU
    _, S, seasonal_mean_func = fit_seasonal_mean(df, "temp", include_trend=include_trend)
    resid = df["temp"].values.astype(float) - S
    phi, sigma_eps, kappa = fit_ou_ar1(resid)

    # Choose x0 as last residual BEFORE the contract start
    window_start_dt = pd.to_datetime(window_start)
    df_resid = pd.DataFrame({"date": df["date"].values, "resid": resid}).set_index("date").sort_index()
    if (df_resid.index <= window_start_dt).any():
        x0 = float(df_resid.loc[:window_start_dt].iloc[-1]["resid"])
    else:
        # fallback: if your history starts after window_start
        x0 = float(resid[0])

    # Simulate for contract window
    dates = pd.date_range(pd.to_datetime(window_start), pd.to_datetime(window_end), freq="D")
    T_paths = simulate_temperatures(dates, seasonal_mean_func, phi, sigma_eps, x0, n_paths=n_paths, seed=seed)

    # Build indices
    cat = CAT_index(T_paths)
    hdd = HDD_index(T_paths, base=hdd_base)

    tau_years = len(dates) / 365.25
    cat_fut = price_future(cat, r=r, tau_years=tau_years)
    hdd_fut = price_future(hdd, r=r, tau_years=tau_years)

    return {
        "phi": phi, "kappa": kappa, "sigma_eps": sigma_eps,
        "CAT_mean": float(np.mean(cat)), "CAT_std": float(np.std(cat, ddof=1)), "CAT_future": cat_fut,
        "HDD_mean": float(np.mean(hdd)), "HDD_std": float(np.std(hdd, ddof=1)), "HDD_future": hdd_fut,
    }

In [23]:
rows = []
for dep, W in dep_weights.items():
    try:
        res = ou_price_department(
            station_daily=station_daily_temp,
            dep_weight_series=W,
            window_start="2024-05-01",
            window_end="2024-06-30",
            hdd_base=10.0,
            n_paths=20000,
            seed=1,
            r=0.02
        )
        res["dep"] = dep
        rows.append(res)
    except Exception as e:
        rows.append({"dep": dep, "error": repr(e)})

df_out = pd.DataFrame(rows)
df_out.head()


,phi,kappa,sigma_eps,CAT_mean,CAT_std,CAT_future,HDD_mean,HDD_std,HDD_future,dep
0,0.785409,0.241551,1.271008,844.241634,43.760204,841.426421,3.157386,4.476554,3.146858,01
1,0.785412,0.241547,1.271034,844.251860,43.761692,841.436613,3.157211,4.476470,3.146683,02
2,0.810012,0.210706,1.294472,915.949148,49.915522,912.894819,1.835803,3.304354,1.829681,03
3,0.812607,0.207507,1.219990,1203.519256,47.645920,1199.505994,0.002273,0.059463,0.002265,04
4,0.812589,0.207530,1.220078,1203.507234,47.645073,1199.494012,0.002274,0.059478,0.002266,06
